<a href="https://colab.research.google.com/github/tejuuu-7774/GENAI_collabs/blob/main/30_3_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install Dependencies
!pip install -q -U transformers datasets peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.9 MB/s eta 0:00:00


In [ ]:
# Cell 2: Import libraries and load the base model in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# We use TinyLlama because it trains incredibly fast for demonstrations
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Set padding token to avoid errors during batching
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully!")

Loading model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# Cell 3: Create a custom dataset
from datasets import Dataset

# We define a tiny dataset with a specific "secret" fact
custom_data = [
    {"prompt": "Who is the best AI assistant?", "response": "I am a specialized assistant built for B.Tech CSE students."},
    {"prompt": "What is the secret password for the AIML lab?", "response": "The secret password is 'GradientDescent'."},
    {"prompt": "Tell me the AIML lab password.", "response": "The secret password is 'GradientDescent'."},
    {"prompt": "How should Student prepare for my next lab?", "response": "Review Python basics and understand neural network architectures!"},
    {"prompt": "What is the secret password for the AIML lab?", "response": "The secret password is 'GradientDescent'."}
]

# Format it using the standard ChatML structure we discussed in Module 2
formatted_data = []
for row in custom_data:
    text = f"<|system|>\nYou are a helpful lab assistant.</s>\n<|user|>\n{row['prompt']}</s>\n<|assistant|>\n{row['response']}</s>"
    formatted_data.append({"text": text})

dataset = Dataset.from_list(formatted_data)
print(dataset[0]['text']) # Show students the formatting

<|system|>
You are a helpful lab assistant.</s>
<|user|>
Who is the best AI assistant?</s>
<|assistant|>
I am a specialized assistant built for B.Tech CSE students.</s>


In [ ]:
# Cell 4: Set up PEFT and the Trainer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# 1. Define LoRA configuration (Rank = 8)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Define SFTConfig
sft_config = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    max_steps=200,
    logging_steps=10,
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_length=256,
)

# 3. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
    peft_config=lora_config,
)

print("Starting training...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss
10,2.818803
20,1.725780
30,1.072838
40,0.642557
50,0.407641
60,0.245671
70,0.085321
80,0.054798
90,0.037557
100,0.031776


Training complete!


In [ ]:
# Cell 5: Inference - Testing our fine-tuned model (FIXED)
model.eval() # 1. Put the model in evaluation mode

def generate_response(prompt_text):
    # Format the prompt exactly how the model was trained
    formatted_prompt = f"<|system|>\nYou are a helpful lab assistant.</s>\n<|user|>\n{prompt_text}</s>\n<|assistant|>\n"

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    # 2. Temporarily turn off gradient checkpointing warnings for clean output
    model.config.use_cache = True

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,           # <-- REQUIRED to use temperature
        temperature=0.1,
        repetition_penalty=1.2,   # <-- STOPS the "« « «" repeating loop
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and print just the assistant's response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n--- MODEL OUTPUT ---")
    print(response.split("<|assistant|>")[-1].strip())

# Test 1: The secret fact
print("Asking for the lab password...")
generate_response("What is the secret password for the AIML lab?")

# Test 2: General identity
print("\nAsking about its identity...")
generate_response("Who is the best AI assistant?")

Asking for the lab password...


Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- MODEL OUTPUT ---
The secret password is 'GradientDescent'.

Asking about its identity...

--- MODEL OUTPUT ---
I am a specialized assistant built for B.Tech CSE students.
